# AdaptRoute — LoRA Adapter Training (All 4 Domains)

Trains one LoRA adapter per domain on top of a **frozen, 4-bit NF4 quantised Qwen2.5-1.5B** base model,
then pushes each adapter to HuggingFace Hub.

**New architecture:** `Query → Firewall → Gate → Adapter merge → Response`

| Domain | HF Dataset | Adapter pushed as |
|--------|-----------|-------------------|
| code | `iamtarun/python_code_instructions_18k_alpaca` | `{HF_USERNAME}/code-adaptroute` |
| math | `lighteval/MATH` (via DigitalLearningGmbH/MATH-lighteval) | `{HF_USERNAME}/math-adaptroute` |
| qa | `rajpurkar/squad` | `{HF_USERNAME}/qa-adaptroute` |
| medical | `lavita/ChatDoctor-HealthCareMagic-100k` | `{HF_USERNAME}/medical-adaptroute` |

**All 4 adapters trained sequentially in a single run.  
Expected runtime on A100:** ~25 min per adapter (~100 min total)

In [11]:
!pip install -U bitsandbytes>=0.46.1 trl datasets pyarrow

## 1. Configuration
Set `HF_USERNAME`. Everything else is pre-tuned for Qwen2.5-1.5B on A100.

In [12]:
# ─── USER CONFIG ──────────────────────────────────────────────────────────────
HF_USERNAME   = "kunjcr2"
WANDB_PROJECT = "adaptroute-adapters-v3"

BASE_MODEL    = "Qwen/Qwen2.5-1.5B"
OUTPUT_ROOT   = "./adapters"
SEED          = 42

# Global defaults tuned for quality on A100 40GB with 4-bit QLoRA
N_SAMPLES     = 20_000
MAX_LENGTH    = 1024

# LoRA / QLoRA
LORA_R               = 32
LORA_ALPHA           = 64
LORA_DROPOUT         = 0.05
LORA_TARGET_MODULES  = "all-linear"   # quality-first QLoRA choice

# SFT defaults
BATCH_SIZE      = 8
GRAD_ACCUM      = 4                    # effective batch = 32
NUM_EPOCHS      = 3
LEARNING_RATE   = 8e-5
WARMUP_RATIO    = 0.03
WEIGHT_DECAY    = 0.01
MAX_GRAD_NORM   = 0.3
LOGGING_STEPS   = 10
SAVE_STRATEGY   = "epoch"
EVAL_STRATEGY   = "no"
PACKING         = False                # cleaner supervision than packing=True
# ──────────────────────────────────────────────────────────────────────────────

ADAPTERS = [
    {
        "name": "code",
        "hf_dataset": "iamtarun/python_code_instructions_18k_alpaca",
        "hf_config": None,
        "hf_split": "train",
        "stream": False,
        "num_epochs": 3,
        "max_length": 1024,
        "n_samples": 18_000,
        "learning_rate": 8e-5,
        "lora_r": 32,
        "lora_alpha": 64,
    },
    {
        "name": "math",
        "hf_dataset": "DigitalLearningGmbH/MATH-lighteval",
        "hf_config": None,
        "hf_split": "train",
        "stream": False,
        "num_epochs": 3,
        "max_length": 1024,
        "n_samples": 20_000,
        "learning_rate": 7e-5,
        "lora_r": 32,
        "lora_alpha": 64,
    },
    {
        "name": "qa",
        "hf_dataset": "rajpurkar/squad",
        "hf_config": None,
        "hf_split": "train",
        "stream": False,
        "num_epochs": 2,
        "max_length": 512,
        "n_samples": 20_000,
        "learning_rate": 8e-5,
        "lora_r": 16,
        "lora_alpha": 32,
    },
    {
        "name": "medical",
        "hf_dataset": "lavita/ChatDoctor-HealthCareMagic-100k",
        "hf_config": None,
        "hf_split": "train",
        "stream": False,
        "num_epochs": 2,
        "max_length": 768,
        "n_samples": 16_000,
        "learning_rate": 6e-5,
        "lora_r": 16,
        "lora_alpha": 32,
    },
]

## 2. Authentication

In [13]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("✓ HuggingFace login OK")

✓ HuggingFace login OK


In [14]:
if WANDB_PROJECT:
    import wandb
    wandb.login(key=userdata.get("WANDB_API"))
    print("✓ W&B ready (runs will be started per-adapter)")
else:
    print("Skipping W&B")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


✓ W&B ready (runs will be started per-adapter)


## 3. Dataset Formatters

Each raw dataset has a different schema. These functions normalise every domain
into a single Qwen2.5 chat-formatted string:

```
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>
```

| Domain | Raw fields used | Mapping |
|--------|----------------|--------|
| code | `instruction`, `input`, `output` | instruction + input as user turn, output as response |
| math | `problem`, `solution` | instruction = problem, response = solution |
| qa | `context`, `question`, `answers.text[0]` | instruction = context + question, response = answer |
| medical | `instruction`, `input`, `output` | instruction = instruction + input, response = output |

In [15]:
def make_prompt_completion(user_text: str, assistant_text: str):
    prompt = f"<|im_start|>user\n{user_text.strip()}<|im_end|>\n<|im_start|>assistant\n"
    completion = f"{assistant_text.strip()}<|im_end|>"
    return {"prompt": prompt, "completion": completion}

# ── Per-domain formatters ────────────────────────────────────────────────────
def format_code(record):
    instruction = record.get("instruction", "").strip()
    inp = record.get("input", "").strip()
    output = record.get("output", "").strip()
    if not instruction or not output:
        return None
    user_turn = f"{instruction}\n{inp}" if inp else instruction
    return make_prompt_completion(user_turn, output)

def format_math(record):
    problem = record.get("problem", "").strip()
    solution = record.get("solution", "").strip()
    if not problem or not solution:
        return None
    return make_prompt_completion(problem, solution)

def format_qa(record):
    context = record.get("context", "").strip()
    question = record.get("question", "").strip()
    answers = record.get("answers", {})
    texts = answers.get("text", [])
    if not context or not question or not texts:
        return None
    answer = texts[0].strip()
    user_turn = (
        "Read the passage and answer the question using only the passage.\n\n"
        f"Passage:\n{context}\n\nQuestion: {question}"
    )
    return make_prompt_completion(user_turn, answer)

def format_medical(record):
    instruction = record.get("instruction", "").strip()
    inp = record.get("input", "").strip()
    output = record.get("output", "").strip()
    if not output:
        return None

    # Style control for safer/shorter behavior
    user_turn = f"{instruction}\n{inp}" if inp else instruction
    user_turn = (
        "Give a concise, calm, safety-aware medical response. "
        "Avoid repetition. State likely concern and immediate next diagnostic/clinical steps.\n\n"
        + user_turn
    )
    return make_prompt_completion(user_turn, output)

FORMATTERS = {
    "code"    : format_code,
    "math"    : format_math,
    "qa"      : format_qa,
    "medical" : format_medical,
}

print("✓ Formatters defined")

# ── Quick sanity-check with a dummy record ───────────────────────────────────
dummy_math = {"problem": "What is 2+2?", "solution": "4", "level": "Level 1", "type": "Algebra"}
print("\nSample math record:")
print(format_math(dummy_math))

✓ Formatters defined

Sample math record:
{'prompt': '<|im_start|>user\nWhat is 2+2?<|im_end|>\n<|im_start|>assistant\n', 'completion': '4<|im_end|>'}


## 4. Load Base Model (once, shared across all adapters)

The base model is loaded **once** in 4-bit NF4 and kept frozen throughout.
Each adapter training run starts fresh from this same frozen base.

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit               = True,
    bnb_4bit_quant_type        = "nf4",
    bnb_4bit_compute_dtype     = torch.bfloat16,
    bnb_4bit_use_double_quant  = True,
)

print(f"Loading {BASE_MODEL} in 4-bit NF4 ...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # required for causal LM training

total_params = sum(p.numel() for p in base_model.parameters())
print(f"✓ Loaded | Total params: {total_params/1e9:.2f}B")

Loading Qwen/Qwen2.5-1.5B in 4-bit NF4 ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✓ Loaded | Total params: 0.89B


## 5. Training Loop — All 4 Adapters

For each domain:
1. Stream / load the raw HF dataset
2. Format records using the domain formatter
3. Attach a fresh LoRA adapter to the frozen base
4. Run SFT with `SFTTrainer`
5. Save adapter locally
6. Push to `{HF_USERNAME}/{domain}-adaptroute`
7. Detach the adapter so the base is clean for the next domain

In [17]:
import gc
import random
import json
from pathlib import Path

import wandb
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from huggingface_hub import HfApi
from transformers import TrainingArguments

random.seed(SEED)
api = HfApi()


def load_and_format(adapter_cfg: dict, formatter, n: int) -> Dataset:
    hf_ds   = adapter_cfg["hf_dataset"]
    cfg     = adapter_cfg["hf_config"]
    split   = adapter_cfg["hf_split"]
    stream  = adapter_cfg["stream"]

    if stream:
        raw = load_dataset(hf_ds, name=cfg, split=split, streaming=True, trust_remote_code=True)
        records = []
        for rec in raw:
            if len(records) >= n * 3:
                break
            records.append(rec)
    else:
        kwargs = {"split": split, "trust_remote_code": True}
        if cfg:
            kwargs["name"] = cfg
        raw = load_dataset(hf_ds, **kwargs)
        records = list(raw)

    random.shuffle(records)

    rows = []
    for rec in records:
        formatted = formatter(rec)
        if not formatted:
            continue
        if len(formatted["completion"]) < 8:
            continue
        rows.append(formatted)
        if len(rows) >= n:
            break

    return Dataset.from_list(rows)


def train_adapter(adapter_cfg: dict) -> None:
    name       = adapter_cfg["name"]
    repo_id    = f"{HF_USERNAME}/{name}-adaptroute-v3"
    output_dir = f"{OUTPUT_ROOT}/{name}"
    formatter  = FORMATTERS[name]

    # Get per-adapter overrides or fallback to globals
    adapter_epochs = adapter_cfg.get("num_epochs", NUM_EPOCHS)
    adapter_maxlen = adapter_cfg.get("max_length", MAX_LENGTH)
    n_samples = adapter_cfg.get("n_samples", N_SAMPLES)

    print(f"\n{'='*65}")
    print(f"  TRAINING: {name.upper()} adapter  →  {repo_id}")
    print(f"  Config: Epochs={adapter_epochs}, MaxLen={adapter_maxlen}")
    print(f"{'='*65}")

    # 1. Data
    dataset = load_and_format(adapter_cfg, formatter, n_samples)

    # 2. Fresh LoRA adapter on the frozen base
    lora_cfg = LoraConfig(
        task_type      = TaskType.CAUSAL_LM,
        r              = adapter_cfg.get("lora_r", LORA_R),
        lora_alpha     = adapter_cfg.get("lora_alpha", LORA_ALPHA),
        lora_dropout   = LORA_DROPOUT,
        target_modules = LORA_TARGET_MODULES,
        bias           = "none",
    )
    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()

    # 3. W&B run (optional)
    if WANDB_PROJECT:
        wandb.init(project=WANDB_PROJECT, name=f"{name}-adapter", reinit=True)

    # 4. SFT
    total_steps  = (len(dataset) // (BATCH_SIZE * GRAD_ACCUM)) * adapter_epochs
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

    sft_config = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=adapter_epochs,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=adapter_cfg.get("learning_rate", LEARNING_RATE),
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        bf16=True,
        logging_steps=LOGGING_STEPS,
        save_strategy=SAVE_STRATEGY,
        eval_strategy=EVAL_STRATEGY,
        report_to="wandb" if WANDB_PROJECT else "none",
        max_length=adapter_maxlen,
        packing=False,
        seed=SEED,
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=dataset,
        processing_class=tokenizer,
    )

    trainer.train()

    # 5. Save adapter only
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"  ✓ Adapter saved locally → {output_dir}")

    # 6. Write model card
    card = f"""---
base_model: {BASE_MODEL}
library_name: peft
license: apache-2.0
tags:
  - lora
  - peft
  - adaptroute
  - {name}
---

# {name}-adaptroute

LoRA adapter for the **{name}** domain in [AdaptRoute](https://adaptroute.vercel.app).

Mounted onto a frozen 4-bit NF4 quantised `{BASE_MODEL}` at inference time via
`peft.add_weighted_adapter()` — weights provided by the gating network.

## LoRA Config
- r = {LORA_R}, alpha = {LORA_ALPHA}, dropout = {LORA_DROPOUT}
- Target modules: {LORA_TARGET_MODULES}
- Training: {adapter_epochs} epochs on {n_samples} samples, lr={LEARNING_RATE}, max_len={adapter_maxlen}

## Training Data
- Source: `{adapter_cfg['hf_dataset']}`
"""
    Path(f"{output_dir}/README.md").write_text(card)

    # 7. Push adapter to Hub
    api.create_repo(repo_id=repo_id, exist_ok=True, token=HF_TOKEN)
    api.upload_folder(
        folder_path    = output_dir,
        repo_id        = repo_id,
        token          = HF_TOKEN,
        commit_message = f"Add {name}-adaptroute LoRA adapter",
    )
    print(f"  ✓ Pushed → https://huggingface.co/{repo_id}-v3")

    # 8. Detach LoRA so base_model is clean for next domain
    if WANDB_PROJECT:
        wandb.finish()
    del model, trainer, dataset
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  ✓ VRAM cleared\n")


print("✓ Training functions defined — run the next cell to start")

✓ Training functions defined — run the next cell to start


## 6. Run — Train All 4 Adapters

This single cell trains and pushes all adapters sequentially.
To train only one domain, replace `ADAPTERS` with e.g. `[ADAPTERS[0]]`.

In [ ]:
for adapter_cfg in ADAPTERS[1:]:
    train_adapter(adapter_cfg)

print("\n" + "="*65)
print("ALL ADAPTERS TRAINED AND PUSHED")
print("="*65)
for a in ADAPTERS:
    print(f"  https://huggingface.co/{HF_USERNAME}/{a['name']}-adaptroute")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'iamtarun/python_code_instructions_18k_alpaca' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'iamtarun/python_code_instructions_18k_alpaca' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



  TRAINING: CODE adapter  →  kunjcr2/code-adaptroute-v3
  Config: Epochs=3, MaxLen=1024
trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/18000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/18000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.843024
20,0.804993
30,0.656409
40,0.745492
50,0.606946
60,0.642062
70,0.647068
80,0.657444
90,0.593135
100,0.686010


## 7. Verify — Smoke Test All Adapters from Hub

Loads each adapter back from the Hub and runs a forward pass to confirm
they mount correctly on the base model.

In [ ]:
from peft import PeftModel

# More challenging prompts to test the limits of the specialized adapters
VERIFY_PROMPTS = {
    "code": "Write a Python function that takes a list of dictionaries (representing products with 'name', 'price', and 'category') and returns the name of the most expensive product in a specific category. Handle the case where the category is empty.",
    "math": "A tank has two pipes. Pipe A fills the tank in 5 hours, and Pipe B empties it in 8 hours. If both pipes are opened at the same time when the tank is empty, how long will it take to fill the tank to 75% capacity? Show your steps.",
    "qa": "Read the following passage and answer the question.\n\nPassage:\nQuantum entanglement is a physical phenomenon that occurs when a group of particles are generated, interact, or share spatial proximity in a way such that the quantum state of each particle of the group cannot be described independently of the state of the others, including when the particles are separated by a large distance. The topic of quantum entanglement is at the heart of the disparity between classical and quantum physics: entanglement is a primary feature of quantum mechanics lacking in classical mechanics.\n\nQuestion: Based on the passage, what is the fundamental difference between classical and quantum physics regarding the description of particle states?",
    "medical": "Patient Case: A 55-year-old male presents with a sudden onset of 'the worst headache of my life', accompanied by photophobia and neck stiffness. He has no prior history of migraines. What is the most likely differential diagnosis, and what immediate diagnostic steps should be taken?",
}

print("Verifying adapters from Hub with tougher prompts ...\n")

for a in ADAPTERS:
    name    = a["name"]
    # Ensure we point to the latest -v2 versions if that's what was pushed
    repo_id = f"{HF_USERNAME}/{name}-adaptroute-v3"
    prompt  = VERIFY_PROMPTS[name]

    print(f"── {name.upper()} ({repo_id}) ──")
    try:
        # Load the adapter onto the frozen base_model
        peft_model = PeftModel.from_pretrained(base_model, repo_id)
        peft_model.eval()

        enc = tokenizer(
            f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n",
            return_tensors="pt",
        ).to(base_model.device)

        with torch.no_grad():
            out = peft_model.generate(
                **enc,
                max_new_tokens = 200, # Increased for tougher questions
                do_sample      = False,
                temperature    = None,
                top_p          = None,
            )

        response = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        # Clean up common hallucinations/headers if needed
        response = response.replace("ChatDoctor", "AdaptRoute").replace("Chat Doctor", "AdaptRoute")

        print(f"Prompt  : {prompt}...")
        print(f"Response:\n{response}")
        print(f"✓ OK\n")

        # Cleanup VRAM for the next adapter
        del peft_model
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"✗ FAILED: {e}\n")

print("Verification complete.")

Verifying adapters from Hub with tougher prompts ...

── CODE (kunjcr2/code-adaptroute-v2) ──
Prompt  : Write a Python function that takes a list of dictionaries (representing products with 'name', 'price', and 'category') and returns the name of the most expensive product in a specific category. Handle the case where the category is empty....
Response:
def most_expensive_product(products, category):
    if not category:
        return None
    max_price = 0
    max_product = None
    for product in products:
        if product['category'] == category:
            if product['price'] > max_price:
                max_price = product['price']
                max_product = product
    return max_product['name'] if max_product else None

products = [
    {'name': 'Product A', 'price': 10, 'category': 'Electronics'},
    {'name': 'Product B', 'price': 20, 'category': 'Electronics'},
    {'name': 'Product C', 'price': 30, 'category': 'Clothing'},
    {'name': 'Product D', 'price': 40, 'categ

## 8. Soft Merge Helper (AdaptRoute inference)

This is the function your main pipeline calls after the gate returns weights.
Paste it into `adapters/merge_adapters.py`.

In [ ]:
from peft import PeftModel


def load_adapter(base, repo_id: str):
    """Load a single LoRA adapter from the Hub onto the base model."""
    return PeftModel.from_pretrained(base, repo_id)


def soft_merge_and_generate(
    base_model,
    tokenizer,
    gate_result: dict,
    max_new_tokens: int = 256,
) -> str:
    """
    Takes the output of gate() and runs a soft-merged forward pass.

    gate_result expected shape:
      {
        'blocked'      : False,
        'top_adapters' : ['lora-code', 'lora-math'],   # adapter names
        'top_weights'  : [0.72, 0.28],                 # normalised
        'hard_routed'  : False,
        'probs'        : {...}
      }

    Adapter name → HF repo mapping:
      lora-{domain} → {HF_USERNAME}/{domain}-adaptroute
    """
    if gate_result["blocked"]:
        return "[BLOCKED] This request was identified as a prompt injection attempt."

    query        = gate_result.get("query", "")   # attach query to gate_result upstream
    top_adapters = gate_result["top_adapters"]     # e.g. ['lora-code', 'lora-math']
    top_weights  = gate_result["top_weights"]

    # Map lora-{domain} → HF repo IDs
    adapter_repo_ids = [
        f"{HF_USERNAME}/{a.replace('lora-', '')}-adaptroute"
        for a in top_adapters
    ]

    if len(adapter_repo_ids) == 1 or gate_result.get("hard_routed"):
        # Hard routing — single adapter, no merge needed
        model = PeftModel.from_pretrained(base_model, adapter_repo_ids[0])
    else:
        # Soft routing — load first adapter, then merge in remaining with weights
        model = PeftModel.from_pretrained(
            base_model, adapter_repo_ids[0], adapter_name=top_adapters[0]
        )
        for repo_id, name in zip(adapter_repo_ids[1:], top_adapters[1:]):
            model.load_adapter(repo_id, adapter_name=name)

        model.add_weighted_adapter(
            adapters         = top_adapters,
            weights          = top_weights,
            adapter_name     = "merged",
            combination_type = "linear",
        )
        model.set_adapter("merged")

    model.eval()
    prompt = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
    enc    = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = None,
            top_p          = None,
        )

    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)


print("✓ soft_merge_and_generate() defined")
print("  Copy this cell into adapters/merge_adapters.py for use in the main pipeline.")

✓ soft_merge_and_generate() defined
  Copy this cell into adapters/merge_adapters.py for use in the main pipeline.
